# SDA-DSC-212 — Time Series Analysis and Forecasting
## Lab 5 — Calibrated Day-Ahead Intervals

**Duration** 50 minutes · **Work in pairs** · `git checkout lab5-start`

> The one rule for this whole course: every feature, every split and every scaler
> must be computable using only data that existed at the forecast origin.

### Tasks

| min | task |
|---|---|
| 5 | Find the **two bugs** in the starter interval you inherited. Note both in `INTERVALS.md`. |
| 10 | Reuse the Lab 4 feature matrix. Split train / calibration / test **by time**. Fit `QuantileLGBM` at τ ∈ {0.05, 0.50, 0.95} and repair quantile crossing. |
| 10 | Coverage and mean width of the raw band on the test window. It will come out **below** 90%. |
| 10 | Calibrate with `CQR` on the held-out October window. Re-apply. Coverage should snap to ≈ 0.90. |
| 10 | Report coverage **marginally and conditionally** with `coverage_report`. Pinball losses and the Winkler score. Plot one calibrated day. |
| 5 | Recommend one method in `INTERVALS.md`. Commit. |

> **Day boundary.** Tasks 1–3 close Day 2; the room goes home holding an interval that
> visibly under-covers. Tasks 4–6 open Day 3.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

DATA = "../data"

In [ ]:
from tayyar.data.load import load_demand, load_calendar, join_calendar
from tayyar.features.build import build_features

df = join_calendar(load_demand(f"{DATA}/ksa_grid_demand.csv"),
                   load_calendar(f"{DATA}/ksa_calendar.csv"))
feat = build_features(df[["demand_mw", "temp_c", "is_weekend",
                          "is_ramadan", "is_eid", "is_national_day"]])
Xcols = [c for c in feat.columns if c != "demand_mw"]
X, yy = feat[Xcols].astype(float), feat["demand_mw"]

CUT_TRAIN, CUT_CAL = "2023-09-30 23:00", "2023-10-31 23:00"
print(f"{len(feat):,} rows x {len(Xcols)} features (the Lab 4 design matrix)")

### Task 1 — the starter interval you inherited

Run the next cell. It produces a 90% band and a coverage number. Five minutes: name the two
bugs precisely, and write them into `INTERVALS.md`. Do not fix them yet.

In [ ]:
# ---------------------------------------------------------------------------
# The starter code you inherited
#
# "Day-ahead point forecast with a 90% interval. I checked the coverage."
#
# It runs and it produces an interval. Two lines make the reported coverage
# worthless — one about WHAT is being estimated, one about WHERE it is measured.
# ---------------------------------------------------------------------------
import lightgbm as lgb

Y24 = yy.shift(-24)                                  # the day-ahead target
ok = X.notna().all(axis=1) & Y24.notna()
Xfit, Yfit = X[ok][:CUT_TRAIN], Y24[ok][:CUT_TRAIN]

point = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.06, num_leaves=63,
                          verbose=-1, random_state=212).fit(Xfit, Yfit)

fitted = point.predict(Xfit)
resid = Yfit - fitted
se = resid.std() / np.sqrt(len(resid))               # "the standard error"
half = 1.645 * se                                    # "90%"

lo_s, hi_s = fitted - half, fitted + half
cov_s = float(np.mean((Yfit >= lo_s) & (Yfit <= hi_s)))

print(f"residual sd            : {resid.std():8,.1f} MW")
print(f"reported 90% band width: {2 * half:8,.1f} MW")
print(f"reported coverage      : {cov_s:8.3f}")

In [ ]:
# TODO (5 min) — do not change the cell above yet.
#
# Bug 1 is about WHAT the interval estimates. Write down, in one line each:
#   - what does resid.std() / sqrt(n) estimate?
#   - what does a control room need an interval around tomorrow's demand to contain?
#   - what happens to that band as n grows to a million rows? Should it?
#
# Bug 2 is about WHERE coverage is measured. Compute the same band's coverage on
# the test window (X[ok][CUT_CAL:]) and compare it with the printed number.
#
# Then separate the two: tabulate coverage on TRAIN and on TEST for
#   (a) half = 1.645 * resid.std() / sqrt(n)     and
#   (b) half = 1.645 * resid.std()
#
# HINT: the two bugs push the reported number in opposite directions, so neither is
#       visible from the printout alone. One factor is sqrt(n); the other is a split.

### Task 2 — three quantiles, split by time, de-crossed

Three time blocks, disjoint and in order:

| block | window | purpose |
|---|---|---|
| train | … through 2023-09-30 | fit the quantile models |
| calibration | October 2023 | estimate q̂ — never used for fitting |
| test | November–December 2023 | the only honest coverage estimate |

`QuantileLGBM` fits one model per (horizon, τ): 3 × 24 = 72 models. Nothing constrains the
three fits to be monotone, so q95 can land below q50 for some rows — an incoherent
distribution that would produce a negative-width interval. Sort each row.

In [ ]:
from tayyar.models.quantile import QuantileLGBM

# TODO (10 min)
#   1. Three disjoint blocks, in time order:
#        Xtr = X[:CUT_TRAIN].dropna();  ytr = yy[Xtr.index]
#        Xca, yca = X[CUT_TRAIN:CUT_CAL], yy[CUT_TRAIN:CUT_CAL]     # October
#        Xte, yte = X[CUT_CAL:],          yy[CUT_CAL:]              # Nov-Dec
#      Print each window and check they do not overlap.
#   2. qm = QuantileLGBM(quantiles=(0.05, 0.50, 0.95), horizon=24).fit(Xtr, ytr)
#      72 models. Start it and read the next paragraph while it runs.
#   3. Write two helpers:
#        q_predict(Xs) -> array (3, n_origins, 24), one plane per quantile
#        day_blocks(Xs, ys) -> the 24 actual hours after each daily origin
#   4. COUNT the quantile crossings before you repair them — separately for
#      (q05 > q50), (q50 > q95) and (q05 > q95). Then repair: np.sort(P, axis=0)
#      takes the order statistics; if the band itself is never inverted you can
#      instead clip the median into [q05, q95] and keep the model's endpoints.
#      Report the largest correction you applied.
#
# HINT: report the crossing count. "We sorted it" without a count is how a broken
#       quantile model gets shipped quietly.

### Task 3 — what the raw band actually covers

In [ ]:
from tayyar.eval.pinball import coverage

# TODO (10 min)
#   lo_raw, med, hi_raw = Pte[0], Pte[1], Pte[2]
#   Report, on the TEST window:
#     - nominal coverage (0.90)
#     - empirical coverage of [lo_raw, hi_raw]
#     - mean width in MW
#   Then the same coverage on the CALIBRATION month.
#
#   Write the two numbers in INTERVALS.md before you go any further.
#
# HINT: it will come out BELOW 0.90, and that is not a bug in your code. Ask why a
#       model that minimised pinball loss in-sample misses its quantiles out of sample,
#       and whether the calibration month shows the same shortfall. If it does, the
#       shortfall is a property of the model and you can correct for it.

### Task 4 — conformal calibration (CQR)

The recipe, in four lines:

1. **Split.** Hold out a calibration window the model never saw — October here.
2. **Score.** For CQR the nonconformity score is `max(lo - y, y - hi)`: how far the truth fell
   *outside* the band. It is negative when the truth was comfortably inside, which is how the
   band can also be made *narrower* when the model is over-cautious.
3. **q̂.** Take the `ceil((n+1)(1-α))`-th smallest score. The `+1` is the finite-sample
   correction — it is what turns an asymptotic statement into a guarantee that holds at any n.
4. **Widen.** The calibrated interval is `[lo - q̂, hi + q̂]`.

CQR keeps the *shape* the quantile model learned — wide on volatile afternoons, narrow at
04:00 — and rescales it. A symmetric split-conformal band around the point forecast would hit
the same marginal coverage with a constant width, and be badly wrong in both directions at
different hours.

In [ ]:
from tayyar.models.conformal import CQR, conformal_quantile

# TODO (10 min)
#   1. cqr = CQR(alpha=0.10).calibrate(Yca.ravel(), Pca[0].ravel(), Pca[2].ravel())
#   2. lo_cal, hi_cal = cqr.interval(lo_raw, hi_raw)
#   3. Recompute q-hat BY HAND and check it matches cqr.qhat_:
#        scores = np.maximum(Pca[0].ravel() - Yca.ravel(), Yca.ravel() - Pca[2].ravel())
#        k = ceil((n + 1) * (1 - alpha));  q-hat = the k-th smallest score
#      Print n, k, k/n and q-hat. Explain the +1 to your pair.
#   4. Coverage and mean width, before and after, on the TEST window.
#
# HINT: coverage should snap to about 0.90 and the width should grow. If coverage
#       overshoots to 0.99 you calibrated on a window from a different regime — check
#       that the calibration month is adjacent to the test window, not a summer month.

### Task 5 — marginal is not conditional

In [ ]:
from tayyar.models.conformal import coverage_report
from tayyar.eval.pinball import pinball_loss, interval_score

# TODO (10 min)
#   1. Build the conditioning variable: horizon 1..24 tiled across the origins,
#        h = np.tile(np.arange(1, 25), len(Yte))
#      and label h in 12..18 as "12:00-18:00 (peak)", everything else "other hours".
#   2. coverage_report(...) AFTER calibration, then BEFORE. Print both.
#   3. pinball_loss at 0.05 / 0.50 / 0.95, and interval_score (Winkler) at alpha=0.10.
#   4. Plot one day: the calibrated band, the raw band, q50, and the actual.
#
#   Then answer in INTERVALS.md: your marginal coverage is ~0.90. What is it on the
#   peak block? Which hours does the grid actually size reserve for?
#
# HINT: conformal calibration guarantees MARGINAL coverage under exchangeability.
#       Nothing in the recipe promises the peak hours anything. Expect the marginal
#       number to look fine and the peak number not to.

### Mini-quiz

Answer these in your pair before you look at the next cell.

1. For a *future observation*, do you need a prediction interval or a confidence interval?
2. What does minimising pinball loss at τ = 0.9 recover?
3. What exactly does split-conformal guarantee, and under what assumption?
4. Why can two independently-fitted quantile models be incoherent?
5. Two intervals both cover 90%. Which one ships?

### Task 6 — `INTERVALS.md` and commit

Write the recommendation, not the method tour. Four short sections:

1. **The two bugs in the starter**, with the mechanism of each in one sentence.
2. **The numbers**: raw quantile band 0.786 coverage at 2,418 MW mean width; after CQR
   0.902 at 3,235 MW; q̂ = 409 MW.
3. **The recommendation**: quantile LightGBM calibrated by CQR on a rolling recent window,
   at α = 0.10. Reason: it keeps the shape of the quantile model (wide where the model is
   genuinely uncertain) and rescales it to hit nominal coverage, with a distribution-free
   finite-sample guarantee, for the cost of one held-out month.
4. **The caveat, in the same paragraph as the recommendation, not a footnote**: marginal
   0.902 hides 0.845 on the 12:00–18:00 peak. Reserve is sized on those hours. Ship the
   conditional table with the headline or ship neither, and move to adaptive conformal
   before this interval is used to size anything.

```
git add -A && git commit -m "feat(models): quantile LightGBM + CQR-calibrated day-ahead intervals"
```